<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Phase56.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 56 FIXED — frozen-method multi-seed robustness, negative controls, severity sweeps, and cohort stability

Phase 55 prepared an external blinded-adjudication package, but external review is not feasible for the current project. Phase 56 therefore strengthens the computational evidence **without changing the frozen Phase 42 method**.

### Phase 56 objectives

1. Verify the exact Phase 39 → 42 → 43 → 55 evidence chain.
2. Reconstruct the Phase 43 generator and confirm the frozen detector reproduces the sealed holdout truth.
3. Freeze **12 new untouched synthetic robustness seeds before generation**.
4. Run the same 229-case challenge design under every seed without retuning.
5. Run additional negative controls that should not trigger fidelity alerts.
6. Characterize the frozen numeric tolerance with a predeclared severity sweep.
7. Quantify downstream cohort-membership consequences across seeds and fault families.
8. Bootstrap uncertainty **by seed cluster**.
9. Rerun one complete seed and verify byte-stable/canonical reproducibility.
10. Record a transparent post-feasibility protocol amendment: external adjudication was not executed; independent external validation remains false.

### Claim boundary

This phase can support **multi-seed synthetic robustness and reproducibility**. It cannot support independent external validation, operational clinical safety, official OMOP/MI-CDM execution, or population-level diagnostic performance.


In [1]:
# CODE CELL 1/11 — Drive, constants, frozen robustness specification

import os,re,csv,json,hashlib,sqlite3,shutil,tempfile,zipfile,time,random,math
import datetime as dt
from pathlib import Path
from copy import deepcopy
from collections import Counter,defaultdict
import statistics

PROJECT='ICHI2027_neuroimaging_fhir_omop_fidelity'
PHASE='56_frozen_method_multiseed_robustness_and_cohort_stability'

EXPECTED_P55_SHAREABLE='3a69736ba1b9afb42c3bf6041b6ecfa0234da438e8800478fa29dd295d80c6af'
EXPECTED_P39_DB='0c9507d8fcf5986e9763d473bf0b342a1c4daaad6c87ee04915986518f5c2384'
EXPECTED_P42_METHOD_FILE='490657f569f36fec6dd7c4b793556490a4c1318249fbb9111999f3150ab149a8'
EXPECTED_P42_METHOD_CONFIG='5bac736739cffa44b1ebbc2ced2886f7e267887a4e78a1b1ba8e012651616902'
EXPECTED_P43_SUMMARY='c24b174761f1e18e92c4d5abf761f7ff5d7c5f09740c8a7e3d0f63a9af50495f'
EXPECTED_P43_LOCK='73a709b83dde4490aeac52ab35b150776e93499ac110a57e255e8cca2c459abc'
EXPECTED_TRUTH_SEAL='6952711786b4b99d37204dbbc70762caabfbdb852d0251be5ef231efaf72609b'

HOLDOUT_SEED=430930
ROBUSTNESS_SEEDS=list(range(560930,560942))   # frozen before generation
BOOTSTRAP_SEED=560999
BOOTSTRAP_REPS=5000
ABS_TOLERANCE_MM3=1e-6

# Predeclared severity sweep around the already-frozen absolute tolerance.
VALUE_DELTAS_MM3=[0.0,0.5e-6,0.9e-6,1.1e-6,1e-4,1e-2,1.0,10.0]

FIELDS=['obs_ref','person_ref','study_ref','biomarker_code','anatomy_source_code','laterality',
        'timepoint','effective_utc','value_mm3','unit_code','algorithm_name','algorithm_version',
        'trace_state','trace_version','device_ref','provenance_ref']

REASON_CODES=[
 'ANATOMY_LATERALITY_MISMATCH','UNIT_MISSING_OR_INCOMPATIBLE','VALUE_MISMATCH',
 'PERSON_LINK_MISMATCH','STUDY_TIMEPOINT_LINK_MISMATCH','MODEL_IDENTITY_LOSS',
 'PROVENANCE_LINK_LOSS','TRACE_STATE_LOSS','DUPLICATE_OBSERVATION',
 'DROPPED_OBSERVATION','UNEXPECTED_OBSERVATION'
]

ROBUSTNESS_SPEC={
 'method_config_sha256':EXPECTED_P42_METHOD_CONFIG,
 'abs_tolerance_mm3':ABS_TOLERANCE_MM3,
 'robustness_seeds':ROBUSTNESS_SEEDS,
 'holdout_design_per_seed':{
   'clean_cases':19,
   'single_fault_families':REASON_CODES,
   'single_fault_cases_per_family':15,
   'valid_cm3_normalization_cases':15,
   'compound_cases':30,
   'total_cases':229,
   'authored_fault_events':240
 },
 'value_severity_deltas_mm3':VALUE_DELTAS_MM3,
 'negative_controls':['EXACT_COPY','ROW_REORDER','VALID_CM3_NORMALIZATION','FLOAT_IDENTITY'],
 'bootstrap_reps':BOOTSTRAP_REPS,
 'bootstrap_cluster':'robustness_seed',
 'protocol_status':'POST_FEASIBILITY_COMPUTATIONAL_ROBUSTNESS_EXTENSION_NO_METHOD_RETUNING'
}
ROBUSTNESS_SPEC_SHA=hashlib.sha256(
    json.dumps(ROBUSTNESS_SPEC,sort_keys=True,separators=(',',':')).encode('utf-8')
).hexdigest()

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''):
            h.update(b)
    return h.hexdigest()

def write_csv(path,rows,fieldnames=None):
    rows=list(rows)
    if fieldnames is None:
        if not rows: raise ValueError('fieldnames required for empty CSV')
        fieldnames=list(rows[0].keys())
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames)
        w.writeheader(); w.writerows(rows)

def canonical_hash(obj):
    return hashlib.sha256(
        json.dumps(obj,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode('utf-8')
    ).hexdigest()

if os.environ.get('PHASE56_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE56_LOCAL_DRIVE_ROOT']).expanduser().resolve()
    ROOTS=[MYDRIVE]
else:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive')
    assert MYDRIVE.is_dir()
    ROOTS=[MYDRIVE]
    sd=Path('/content/drive/Shareddrives')
    if sd.is_dir(): ROOTS.append(sd)

hint=os.environ.get('PHASE56_DRIVE_FOLDER_HINT','').strip()
if hint:
    hp=Path(hint).expanduser()
    if not hp.is_absolute(): hp=MYDRIVE/hp
    if not hp.is_dir(): raise FileNotFoundError(hp)
    ROOTS=[hp.resolve()]

WORK=Path(tempfile.mkdtemp(prefix='ichi2027_phase56_'))
INPUT=WORK/'inputs'; INPUT.mkdir()
STAGE=WORK/'outputs'; STAGE.mkdir()

OUT_BASE=Path(os.environ.get('PHASE56_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase56')))
OUT=OUT_BASE/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')

print('Phase56 output:',OUT)
print('Frozen robustness spec SHA-256:',ROBUSTNESS_SPEC_SHA)
print('Robustness seeds:',ROBUSTNESS_SEEDS)


Mounted at /content/drive
Phase56 output: /content/drive/MyDrive/ICHI2027/Phase56/run_20261001T162938_319424Z
Frozen robustness spec SHA-256: d3dc95396ac3bc27cc1ada9aedcba8b6a4cf5d1e6cf951e047f49d0394438b1a
Robustness seeds: [560930, 560931, 560932, 560933, 560934, 560935, 560936, 560937, 560938, 560939, 560940, 560941]


In [2]:
# CODE CELL 2/11 — Auto-discover and checksum-verify exact evidence chain

REQUIRED={
 'p55':('ICHI2027_Phase55_SHAREABLE','.json',EXPECTED_P55_SHAREABLE),
 'p39_db':('ICHI2027_Phase39_SYNTHETIC_SOURCE_LEDGER','.sqlite',EXPECTED_P39_DB),
 'p42_method':('ICHI2027_Phase42_METHOD_SPEC','.json',EXPECTED_P42_METHOD_FILE),
 'p43_summary':('ICHI2027_Phase43_SHAREABLE','.json',EXPECTED_P43_SUMMARY),
 'p43_lock':('ICHI2027_Phase43_METHOD_LOCK','.json',EXPECTED_P43_LOCK),
}

def name_matches(actual,stem,ext):
    return re.fullmatch(re.escape(stem)+r'(?:-\d+|\s*\(\d+\))?'+re.escape(ext),actual,re.I) is not None

hits={k:[] for k in REQUIRED}
for root in ROOTS:
    for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
        dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
        for fn in files:
            for k,(stem,ext,digest) in REQUIRED.items():
                if name_matches(fn,stem,ext):
                    p=Path(here)/fn
                    try:
                        if sha256(p)==digest:
                            hits[k].append(p)
                    except OSError:
                        pass

missing=[k for k,v in hits.items() if not v]
if missing:
    raise FileNotFoundError('Missing exact checksum-approved inputs: '+', '.join(missing))

P={k:sorted(v,key=str)[0] for k,v in hits.items()}

p55=json.loads(P['p55'].read_text('utf-8'))
p43=json.loads(P['p43_summary'].read_text('utf-8'))
lock43=json.loads(P['p43_lock'].read_text('utf-8'))
METHOD=json.loads(P['p42_method'].read_text('utf-8'))

assert p55['phase']=='55_blinded_external_adjudication_package_and_sealed_scoring_key'
assert p55['independently_blinded_external_validation'] is False
assert p55['reviewer_package_created'] is True

assert p43['phase']=='43_frozen_method_synthetic_holdout_no_retuning_audit'
assert p43['holdout_cases']==229
assert p43['holdout_authored_fault_events']==240
assert p43['method_retuned_after_holdout_generation'] is False
assert lock43['holdout_truth_seal_sha256']==EXPECTED_TRUTH_SEAL
assert lock43['holdout_seed']==HOLDOUT_SEED

method_blob=json.dumps(METHOD,sort_keys=True,separators=(',',':')).encode('utf-8')
assert hashlib.sha256(method_blob).hexdigest()==EXPECTED_P42_METHOD_CONFIG

DB_LOCAL=INPUT/'immutable_source.sqlite'
shutil.copyfile(P['p39_db'],DB_LOCAL)
assert sha256(DB_LOCAL)==EXPECTED_P39_DB

spec_path=STAGE/'ICHI2027_Phase56_ROBUSTNESS_SPEC.json'
spec_path.write_text(json.dumps(ROBUSTNESS_SPEC,sort_keys=True,indent=2)+'\n',encoding='utf-8')
assert canonical_hash(json.loads(spec_path.read_text('utf-8')))==ROBUSTNESS_SPEC_SHA

print('PASS: exact Phase39/42/43/55 chain verified.')
print('PASS: robustness specification written before challenge generation.')


PASS: exact Phase39/42/43/55 chain verified.
PASS: robustness specification written before challenge generation.


In [3]:
# CODE CELL 3/11 — Reconstruct immutable synthetic source reference

c=sqlite3.connect('file:'+str(DB_LOCAL)+'?mode=ro',uri=True)
assert c.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
assert c.execute('PRAGMA foreign_key_check').fetchall()==[]
c.row_factory=sqlite3.Row

STUDIES={r['study_ref']:dict(r) for r in c.execute('SELECT * FROM source_study')}
DEVICES={r['device_ref']:dict(r) for r in c.execute('SELECT * FROM source_device')}
PROVS={r['obs_ref']:dict(r) for r in c.execute('SELECT * FROM source_observation_provenance')}
SOURCE=[]

for raw in c.execute('SELECT * FROM source_observation ORDER BY obs_ref'):
    d=dict(raw)
    dev=DEVICES[d['device_ref']]
    d.update(algorithm_name=dev['algorithm_name'],algorithm_version=dev['algorithm_version'])
    assert PROVS[d['obs_ref']]['provenance_ref']==d['provenance_ref']
    SOURCE.append(d)
c.close()

assert len(SOURCE)==148 and len(STUDIES)==38 and len(DEVICES)==19

DDL=(
    "CREATE TABLE projected_study(study_ref TEXT,person_ref TEXT,study_started TEXT);\n"
    "CREATE TABLE projected_feature(obs_ref TEXT,person_ref TEXT,study_ref TEXT,biomarker_code TEXT,"
    "anatomy_source_code TEXT,laterality TEXT,timepoint TEXT,effective_utc TEXT,value_mm3 REAL,"
    "unit_code TEXT,algorithm_name TEXT,algorithm_version TEXT,trace_state TEXT,trace_version TEXT,"
    "device_ref TEXT,provenance_ref TEXT);\n"
)
INSERT='INSERT INTO projected_feature('+','.join(FIELDS)+') VALUES('+','.join('?' for _ in FIELDS)+')'

SQL_PROXY=(
    "WITH observed AS("
    " SELECT f.*,s.person_ref AS linked_person,s.study_started AS linked_started,"
    " CASE WHEN f.unit_code='mm3' THEN f.value_mm3 WHEN f.unit_code='cm3' THEN f.value_mm3*1000.0 ELSE NULL END AS normalized_mm3"
    " FROM projected_feature f LEFT JOIN projected_study s ON f.study_ref=s.study_ref"
    " WHERE f.biomarker_code='hippocampus-volume'"
    "),valid AS("
    " SELECT *,CASE WHEN linked_person=person_ref AND linked_started=effective_utc AND normalized_mm3>0"
    " AND provenance_ref IS NOT NULL"
    " AND ((laterality='LEFT' AND anatomy_source_code='SYN-HIP-LEFT')"
    " OR (laterality='RIGHT' AND anatomy_source_code='SYN-HIP-RIGHT'))"
    " AND timepoint IN ('baseline','12-month') THEN 1 ELSE 0 END AS valid_flag"
    " FROM observed"
    "),slots AS("
    " SELECT person_ref,timepoint,laterality,COUNT(*) AS n,SUM(valid_flag) AS nvalid"
    " FROM valid GROUP BY person_ref,timepoint,laterality"
    "),eligible AS("
    " SELECT person_ref FROM slots GROUP BY person_ref"
    " HAVING COUNT(*)=4 AND MIN(n)=1 AND MAX(n)=1 AND SUM(nvalid)=4"
    ") SELECT person_ref FROM eligible ORDER BY person_ref"
)

def cohort(rows):
    cx=sqlite3.connect(':memory:')
    try:
        cx.executescript(DDL)
        cx.executemany('INSERT INTO projected_study VALUES (?,?,?)',
            [(s['study_ref'],s['person_ref'],s['study_started']) for s in STUDIES.values()])
        cx.executemany(INSERT,[tuple(r.get(k) for k in FIELDS) for r in rows])
        return {z[0] for z in cx.execute(SQL_PROXY)}
    finally:
        cx.close()

SOURCE_COHORT=cohort(SOURCE)
ALL_PEOPLE=sorted({r['person_ref'] for r in SOURCE})
ELIGIBLE=sorted(SOURCE_COHORT)
HIP={p:[r['obs_ref'] for r in SOURCE
       if r['person_ref']==p and r['biomarker_code']=='hippocampus-volume']
     for p in ALL_PEOPLE}

assert len(ALL_PEOPLE)==19
assert len(ELIGIBLE)==15

print('PASS: immutable source reconstructed.')
print('Source people:',len(ALL_PEOPLE),'source cohort:',len(SOURCE_COHORT))


PASS: immutable source reconstructed.
Source people: 19 source cohort: 15


In [4]:
# CODE CELL 4/11 — Frozen detector implementation and exact Phase43 reproduction check

def rows_by_obs(rows):
    d=defaultdict(list)
    for r in rows:
        d[r['obs_ref']].append(r)
    return d

def normalized_mm3(row):
    unit=row.get('unit_code')
    val=row.get('value_mm3')
    if val is None:
        return None
    try:
        val=float(val)
    except Exception:
        return None
    if unit=='mm3':
        return val
    if unit=='cm3':
        return val*1000.0
    return None

def frozen_detect(reference,projected):
    ref=rows_by_obs(reference)
    pro=rows_by_obs(projected)
    alerts=set()
    normalizations=set()

    for obs in sorted(set(ref)|set(pro)):
        rr=ref.get(obs,[])
        pp=pro.get(obs,[])

        if not rr and pp:
            alerts.add((obs,'UNEXPECTED_OBSERVATION'))
            continue
        if rr and not pp:
            alerts.add((obs,'DROPPED_OBSERVATION'))
            continue
        if len(pp)>len(rr):
            alerts.add((obs,'DUPLICATE_OBSERVATION'))
        if len(pp)<len(rr):
            alerts.add((obs,'DROPPED_OBSERVATION'))

        if not rr or not pp:
            continue

        a=rr[0]; b=pp[0]

        if (
            a.get('biomarker_code')!=b.get('biomarker_code')
            or a.get('anatomy_source_code')!=b.get('anatomy_source_code')
            or a.get('laterality')!=b.get('laterality')
        ):
            alerts.add((obs,'ANATOMY_LATERALITY_MISMATCH'))

        av=normalized_mm3(a)
        bv=normalized_mm3(b)
        if bv is None:
            alerts.add((obs,'UNIT_MISSING_OR_INCOMPATIBLE'))
        elif av is not None:
            if a.get('unit_code')!=b.get('unit_code') and abs(av-bv)<=ABS_TOLERANCE_MM3:
                normalizations.add(obs)
            if abs(av-bv)>ABS_TOLERANCE_MM3:
                alerts.add((obs,'VALUE_MISMATCH'))

        if a.get('person_ref')!=b.get('person_ref'):
            alerts.add((obs,'PERSON_LINK_MISMATCH'))
            alerts.add((obs,'STUDY_TIMEPOINT_LINK_MISMATCH'))

        if (
            a.get('study_ref')!=b.get('study_ref')
            or a.get('timepoint')!=b.get('timepoint')
            or a.get('effective_utc')!=b.get('effective_utc')
        ):
            alerts.add((obs,'STUDY_TIMEPOINT_LINK_MISMATCH'))

        if (
            a.get('algorithm_name')!=b.get('algorithm_name')
            or a.get('algorithm_version')!=b.get('algorithm_version')
        ):
            alerts.add((obs,'MODEL_IDENTITY_LOSS'))

        if (
            a.get('device_ref')!=b.get('device_ref')
            or a.get('provenance_ref')!=b.get('provenance_ref')
        ):
            alerts.add((obs,'PROVENANCE_LINK_LOSS'))

        if (
            a.get('trace_state')!=b.get('trace_state')
            or a.get('trace_version')!=b.get('trace_version')
        ):
            alerts.add((obs,'TRACE_STATE_LOSS'))

    return alerts,normalizations

def row_by_id(rows,obs):
    hit=[r for r in rows if r['obs_ref']==obs]
    assert len(hit)==1,obs
    return hit[0]

def inject(f,kind,obs,all_people):
    if kind=='ANATOMY_LATERALITY_MISMATCH':
        t=row_by_id(f['projected'],obs)
        t['anatomy_source_code']='SYN-HIP-RIGHT' if t['anatomy_source_code']=='SYN-HIP-LEFT' else 'SYN-HIP-LEFT'
        f['truth'].add((obs,kind))
    elif kind=='UNIT_MISSING_OR_INCOMPATIBLE':
        row_by_id(f['projected'],obs)['unit_code']='mL'
        f['truth'].add((obs,kind))
    elif kind=='VALUE_MISMATCH':
        t=row_by_id(f['projected'],obs)
        t['value_mm3']=float(t['value_mm3'])*1.01+1.0
        f['truth'].add((obs,kind))
    elif kind=='PERSON_LINK_MISMATCH':
        t=row_by_id(f['projected'],obs)
        t['person_ref']=next(p for p in all_people if p!=t['person_ref'])
        f['truth'].add((obs,'PERSON_LINK_MISMATCH'))
        f['truth'].add((obs,'STUDY_TIMEPOINT_LINK_MISMATCH'))
    elif kind=='STUDY_TIMEPOINT_LINK_MISMATCH':
        row_by_id(f['projected'],obs)['effective_utc']='2099-01-01T00:00:00Z'
        f['truth'].add((obs,kind))
    elif kind=='MODEL_IDENTITY_LOSS':
        row_by_id(f['projected'],obs)['algorithm_name']=None
        f['truth'].add((obs,kind))
    elif kind=='PROVENANCE_LINK_LOSS':
        row_by_id(f['projected'],obs)['device_ref']='Device/missing'
        f['truth'].add((obs,kind))
    elif kind=='TRACE_STATE_LOSS':
        row_by_id(f['reference'],obs)['trace_state']='REVIEW_REQUIRED'
        row_by_id(f['projected'],obs)['trace_version']=None
        f['truth'].add((obs,kind))
    elif kind=='DUPLICATE_OBSERVATION':
        f['projected'].append(deepcopy(row_by_id(f['projected'],obs)))
        f['truth'].add((obs,kind))
    elif kind=='DROPPED_OBSERVATION':
        f['projected']=[r for r in f['projected'] if r['obs_ref']!=obs]
        f['truth'].add((obs,kind))
    elif kind=='UNEXPECTED_OBSERVATION':
        t=deepcopy(row_by_id(f['projected'],obs))
        t['obs_ref']='Observation/phase43-unexpected-'+f['case_id']
        f['projected'].append(t)
        f['truth'].add((t['obs_ref'],kind))
    else:
        raise AssertionError(kind)

def make_case(case_id,kind,person):
    return {
        'case_id':case_id,
        'kind':kind,
        'person_ref':person,
        'reference':deepcopy(SOURCE),
        'projected':deepcopy(SOURCE),
        'truth':set(),
        'expected_normalizations':set()
    }


def generate_seed_cases(seed,prefix='robust',exact_phase43_ids=False):
    rng=random.Random(seed)
    cases=[]

    def next_case_id():
        idx=len(cases)
        if exact_phase43_ids:
            return f'holdout_{idx:04d}'
        return f'{prefix}_{seed}_{idx:04d}'

    for p in ALL_PEOPLE:
        cases.append(make_case(next_case_id(),'CLEAN',p))

    for p in ELIGIBLE:
        for fam in REASON_CODES:
            f=make_case(next_case_id(),fam,p)
            inject(f,fam,rng.choice(HIP[p]),ALL_PEOPLE)
            cases.append(f)

    for p in ELIGIBLE:
        f=make_case(next_case_id(),'VALID_CM3_NORMALIZATION',p)
        obs=rng.choice(HIP[p])
        t=row_by_id(f['projected'],obs)
        t['unit_code']='cm3'
        t['value_mm3']=float(t['value_mm3'])/1000.0
        f['expected_normalizations'].add(obs)
        cases.append(f)

    for i in range(30):
        p=ELIGIBLE[i%len(ELIGIBLE)]
        obs1,obs2=rng.sample(HIP[p],2)
        f=make_case(next_case_id(),'COMPOUND_HOLDOUT',p)
        if i%2==0:
            inject(f,'VALUE_MISMATCH',obs1,ALL_PEOPLE)
            inject(f,'PROVENANCE_LINK_LOSS',obs2,ALL_PEOPLE)
        else:
            inject(f,'UNIT_MISSING_OR_INCOMPATIBLE',obs1,ALL_PEOPLE)
            inject(f,'DROPPED_OBSERVATION',obs2,ALL_PEOPLE)
        cases.append(f)

    assert len(cases)==229
    assert sum(len(f['truth']) for f in cases)==240
    return cases

# Reproduce original Phase43 seed before any new-seed analysis.
# Historical case IDs and unexpected-observation IDs must be byte-for-byte identical
# because the sealed truth hash includes those identifiers.
phase43_cases=generate_seed_cases(
    HOLDOUT_SEED,
    prefix='holdout',
    exact_phase43_ids=True
)

seal_rows=[
    {
      'case_id':f['case_id'],
      'kind':f['kind'],
      'person_ref':f['person_ref'],
      'truth_events':sorted([list(x) for x in f['truth']]),
      'expected_normalizations':sorted(f['expected_normalizations']),
      'projected_rows':len(f['projected'])
    }
    for f in phase43_cases
]

phase43_rebuilt_seal=canonical_hash(seal_rows)
print('Rebuilt Phase43 truth seal:',phase43_rebuilt_seal)
print('Expected Phase43 truth seal:',EXPECTED_TRUTH_SEAL)
assert phase43_rebuilt_seal==EXPECTED_TRUTH_SEAL

truth_total=0; matched_total=0; extra_total=0; norm_total=0
for f in phase43_cases:
    alerts,norms=frozen_detect(f['reference'],f['projected'])
    truth_total+=len(f['truth'])
    matched_total+=len(alerts & f['truth'])
    extra_total+=len(alerts-f['truth'])
    norm_total+=len(norms & f['expected_normalizations'])

assert truth_total==240
assert matched_total==240
assert extra_total==0
assert norm_total==15

print('PASS: frozen detector reproduces Phase43 sealed holdout.')
print('Truth events:',truth_total,'matched:',matched_total,'unexpected alerts:',extra_total,'normalizations:',norm_total)


Rebuilt Phase43 truth seal: 6952711786b4b99d37204dbbc70762caabfbdb852d0251be5ef231efaf72609b
Expected Phase43 truth seal: 6952711786b4b99d37204dbbc70762caabfbdb852d0251be5ef231efaf72609b
PASS: frozen detector reproduces Phase43 sealed holdout.
Truth events: 240 matched: 240 unexpected alerts: 0 normalizations: 15


In [5]:
# CODE CELL 5/11 — Run 12 untouched robustness seeds without retuning

seed_rows=[]
family_acc=defaultdict(lambda:{
    'cases':0,'truth_events':0,'matched_truth_events':0,'unexpected_alerts':0,
    'exact_case_matches':0,'normalization_cases':0,'normalizations_matched':0
})
cohort_rows=[]

def jaccard(a,b):
    u=a|b
    return 1.0 if not u else len(a&b)/len(u)

all_seed_case_hashes={}

for seed in ROBUSTNESS_SEEDS:
    cases=generate_seed_cases(seed,prefix='robust')

    s_truth=s_match=s_extra=s_exact=s_norm_expected=s_norm_match=0
    seed_case_summary=[]

    for f in cases:
        alerts,norms=frozen_detect(f['reference'],f['projected'])

        truth=f['truth']
        expected_norms=f['expected_normalizations']

        matched=len(alerts & truth)
        extra=len(alerts-truth)
        exact=(alerts==truth and norms==expected_norms)

        s_truth+=len(truth)
        s_match+=matched
        s_extra+=extra
        s_exact+=int(exact)
        s_norm_expected+=len(expected_norms)
        s_norm_match+=len(norms & expected_norms)

        fam=family_acc[f['kind']]
        fam['cases']+=1
        fam['truth_events']+=len(truth)
        fam['matched_truth_events']+=matched
        fam['unexpected_alerts']+=extra
        fam['exact_case_matches']+=int(exact)
        fam['normalization_cases']+=int(bool(expected_norms))
        fam['normalizations_matched']+=len(norms & expected_norms)

        target_cohort=cohort(f['projected'])
        false_exclusions=SOURCE_COHORT-target_cohort
        false_inclusions=target_cohort-SOURCE_COHORT

        cohort_rows.append({
            'seed':seed,
            'case_kind':f['kind'],
            'false_exclusions':len(false_exclusions),
            'false_inclusions':len(false_inclusions),
            'source_cohort_n':len(SOURCE_COHORT),
            'target_cohort_n':len(target_cohort),
            'jaccard':jaccard(SOURCE_COHORT,target_cohort)
        })

        seed_case_summary.append({
            'kind':f['kind'],
            'truth_n':len(truth),
            'alert_n':len(alerts),
            'matched_n':matched,
            'extra_n':extra,
            'normalization_expected_n':len(expected_norms),
            'normalization_matched_n':len(norms & expected_norms),
            'target_cohort_n':len(target_cohort)
        })

    seed_rows.append({
        'seed':seed,
        'cases':len(cases),
        'truth_events':s_truth,
        'matched_truth_events':s_match,
        'unexpected_alerts':s_extra,
        'event_recall':s_match/s_truth if s_truth else 1.0,
        'unexpected_alerts_per_case':s_extra/len(cases),
        'exact_case_matches':s_exact,
        'exact_case_match_rate':s_exact/len(cases),
        'expected_normalizations':s_norm_expected,
        'matched_normalizations':s_norm_match,
        'normalization_match_rate':s_norm_match/s_norm_expected if s_norm_expected else 1.0
    })

    all_seed_case_hashes[seed]=canonical_hash(seed_case_summary)

print('PASS: all untouched robustness seeds executed.')
print('Total cases:',sum(r['cases'] for r in seed_rows))
print('Total authored fault events:',sum(r['truth_events'] for r in seed_rows))
print('Total matched:',sum(r['matched_truth_events'] for r in seed_rows))
print('Total unexpected alerts:',sum(r['unexpected_alerts'] for r in seed_rows))


PASS: all untouched robustness seeds executed.
Total cases: 2748
Total authored fault events: 2880
Total matched: 2880
Total unexpected alerts: 0


In [6]:
# CODE CELL 6/11 — Negative controls and frozen tolerance severity sweep

negative_rows=[]

# One control set per robustness seed.
for seed in ROBUSTNESS_SEEDS:
    rng=random.Random(seed+900000)

    controls=[]

    exact=make_case(f'neg_{seed}_exact','EXACT_COPY',ELIGIBLE[0])
    controls.append(exact)

    reorder=make_case(f'neg_{seed}_reorder','ROW_REORDER',ELIGIBLE[1])
    rng.shuffle(reorder['projected'])
    controls.append(reorder)

    cm3=make_case(f'neg_{seed}_cm3','VALID_CM3_NORMALIZATION_CONTROL',ELIGIBLE[2])
    obs=rng.choice(HIP[ELIGIBLE[2]])
    t=row_by_id(cm3['projected'],obs)
    t['unit_code']='cm3'
    t['value_mm3']=float(t['value_mm3'])/1000.0
    cm3['expected_normalizations'].add(obs)
    controls.append(cm3)

    identity=make_case(f'neg_{seed}_float','FLOAT_IDENTITY',ELIGIBLE[3])
    obs=rng.choice(HIP[ELIGIBLE[3]])
    t=row_by_id(identity['projected'],obs)
    t['value_mm3']=float(str(float(t['value_mm3'])))
    controls.append(identity)

    for f in controls:
        alerts,norms=frozen_detect(f['reference'],f['projected'])
        negative_rows.append({
            'seed':seed,
            'control':f['kind'],
            'alert_count':len(alerts),
            'expected_normalization_count':len(f['expected_normalizations']),
            'matched_normalization_count':len(norms & f['expected_normalizations']),
            'passes_negative_control':(
                len(alerts)==0 and
                len(norms & f['expected_normalizations'])==len(f['expected_normalizations'])
            )
        })

assert all(r['passes_negative_control'] for r in negative_rows)

severity_rows=[]
severity_persons=ELIGIBLE[:]

for delta in VALUE_DELTAS_MM3:
    detected=0
    total=0

    for i,p in enumerate(severity_persons):
        f=make_case(f'severity_{delta}_{i}','VALUE_SEVERITY_SWEEP',p)
        obs=HIP[p][i % len(HIP[p])]
        t=row_by_id(f['projected'],obs)

        sign=-1.0 if i%2 else 1.0
        t['value_mm3']=float(t['value_mm3']) + sign*delta

        alerts,_=frozen_detect(f['reference'],f['projected'])
        has_value=any(reason=='VALUE_MISMATCH' for _,reason in alerts)
        detected+=int(has_value)
        total+=1

    severity_rows.append({
        'absolute_delta_mm3':delta,
        'frozen_abs_tolerance_mm3':ABS_TOLERANCE_MM3,
        'cases':total,
        'value_mismatch_detected_cases':detected,
        'detection_rate':detected/total,
        'expected_by_frozen_rule':delta>ABS_TOLERANCE_MM3
    })

# Check characterization against the pre-existing tolerance rule, not tune it.
for r in severity_rows:
    if r['expected_by_frozen_rule']:
        assert r['value_mismatch_detected_cases']==r['cases']
    else:
        assert r['value_mismatch_detected_cases']==0

print('PASS: negative controls and predeclared severity sweep completed.')


PASS: negative controls and predeclared severity sweep completed.


In [7]:
# CODE CELL 7/11 — Aggregate family performance and cohort-consequence stability

family_rows=[]

for kind in sorted(family_acc):
    x=family_acc[kind]
    family_rows.append({
        'case_kind':kind,
        'cases':x['cases'],
        'truth_events':x['truth_events'],
        'matched_truth_events':x['matched_truth_events'],
        'unexpected_alerts':x['unexpected_alerts'],
        'event_recall':x['matched_truth_events']/x['truth_events'] if x['truth_events'] else '',
        'unexpected_alert_rate_per_case':x['unexpected_alerts']/x['cases'],
        'exact_case_matches':x['exact_case_matches'],
        'exact_case_match_rate':x['exact_case_matches']/x['cases'],
        'normalization_cases':x['normalization_cases'],
        'normalizations_matched':x['normalizations_matched']
    })

cohort_family=defaultdict(lambda:{
    'cases':0,'cases_with_false_exclusion':0,'cases_with_false_inclusion':0,
    'false_exclusions':0,'false_inclusions':0,'jaccards':[]
})

for r in cohort_rows:
    x=cohort_family[r['case_kind']]
    x['cases']+=1
    x['cases_with_false_exclusion']+=int(r['false_exclusions']>0)
    x['cases_with_false_inclusion']+=int(r['false_inclusions']>0)
    x['false_exclusions']+=r['false_exclusions']
    x['false_inclusions']+=r['false_inclusions']
    x['jaccards'].append(r['jaccard'])

cohort_family_rows=[]
for kind in sorted(cohort_family):
    x=cohort_family[kind]
    cohort_family_rows.append({
        'case_kind':kind,
        'cases':x['cases'],
        'cases_with_false_exclusion':x['cases_with_false_exclusion'],
        'cases_with_false_inclusion':x['cases_with_false_inclusion'],
        'false_exclusions_total':x['false_exclusions'],
        'false_inclusions_total':x['false_inclusions'],
        'false_exclusion_case_rate':x['cases_with_false_exclusion']/x['cases'],
        'false_inclusion_case_rate':x['cases_with_false_inclusion']/x['cases'],
        'mean_jaccard':statistics.mean(x['jaccards']),
        'min_jaccard':min(x['jaccards'])
    })

print('PASS: family-level detector and cohort consequences aggregated.')


PASS: family-level detector and cohort consequences aggregated.


In [8]:
# CODE CELL 8/11 — Seed-cluster bootstrap uncertainty

rng=random.Random(BOOTSTRAP_SEED)

def percentile(xs,q):
    xs=sorted(xs)
    if not xs:
        return float('nan')
    pos=(len(xs)-1)*q
    lo=math.floor(pos); hi=math.ceil(pos)
    if lo==hi:
        return xs[lo]
    return xs[lo]*(hi-pos)+xs[hi]*(pos-lo)

seed_map={r['seed']:r for r in seed_rows}
seed_ids=ROBUSTNESS_SEEDS[:]

boot_recall=[]
boot_exact=[]
boot_extra=[]
boot_norm=[]

for _ in range(BOOTSTRAP_REPS):
    sampled=[rng.choice(seed_ids) for _ in seed_ids]
    rows=[seed_map[s] for s in sampled]

    truth=sum(r['truth_events'] for r in rows)
    matched=sum(r['matched_truth_events'] for r in rows)
    cases=sum(r['cases'] for r in rows)
    extras=sum(r['unexpected_alerts'] for r in rows)
    exact=sum(r['exact_case_matches'] for r in rows)
    nexp=sum(r['expected_normalizations'] for r in rows)
    nmatch=sum(r['matched_normalizations'] for r in rows)

    boot_recall.append(matched/truth if truth else 1.0)
    boot_exact.append(exact/cases)
    boot_extra.append(extras/cases)
    boot_norm.append(nmatch/nexp if nexp else 1.0)

bootstrap_rows=[]
for name,vals in [
    ('event_recall',boot_recall),
    ('exact_case_match_rate',boot_exact),
    ('unexpected_alerts_per_case',boot_extra),
    ('normalization_match_rate',boot_norm)
]:
    bootstrap_rows.append({
        'metric':name,
        'bootstrap_reps':BOOTSTRAP_REPS,
        'cluster_unit':'robustness_seed',
        'estimate':statistics.mean(vals),
        'ci95_lower':percentile(vals,0.025),
        'ci95_upper':percentile(vals,0.975)
    })

print('PASS: seed-cluster bootstrap completed.')


PASS: seed-cluster bootstrap completed.


In [9]:
# CODE CELL 9/11 — Deterministic reproducibility rerun

REPRO_SEED=ROBUSTNESS_SEEDS[0]

def summarize_seed_for_hash(seed):
    cases=generate_seed_cases(seed,prefix='robust')
    out=[]
    for f in cases:
        alerts,norms=frozen_detect(f['reference'],f['projected'])
        target=cohort(f['projected'])
        out.append({
            'kind':f['kind'],
            'truth':sorted([list(x) for x in f['truth']]),
            'alerts':sorted([list(x) for x in alerts]),
            'expected_normalizations':sorted(f['expected_normalizations']),
            'normalizations':sorted(norms),
            'target_cohort_n':len(target),
            'target_cohort_members':sorted(target)
        })
    return canonical_hash(out)

repro_hash_1=all_seed_case_hashes[REPRO_SEED]
# all_seed_case_hashes intentionally hashes only the compact summary; recompute same compact representation.
def compact_seed_hash(seed):
    cases=generate_seed_cases(seed,prefix='robust')
    rows=[]
    for f in cases:
        alerts,norms=frozen_detect(f['reference'],f['projected'])
        rows.append({
            'kind':f['kind'],
            'truth_n':len(f['truth']),
            'alert_n':len(alerts),
            'matched_n':len(alerts & f['truth']),
            'extra_n':len(alerts-f['truth']),
            'normalization_expected_n':len(f['expected_normalizations']),
            'normalization_matched_n':len(norms & f['expected_normalizations']),
            'target_cohort_n':len(cohort(f['projected']))
        })
    return canonical_hash(rows)

repro_hash_2=compact_seed_hash(REPRO_SEED)
assert repro_hash_1==repro_hash_2

# Stronger full-detail rerun twice.
full_hash_a=summarize_seed_for_hash(REPRO_SEED)
full_hash_b=summarize_seed_for_hash(REPRO_SEED)
assert full_hash_a==full_hash_b

repro={
    'repro_seed':REPRO_SEED,
    'compact_summary_hash_first_run':repro_hash_1,
    'compact_summary_hash_rerun':repro_hash_2,
    'compact_hash_equal':True,
    'full_detail_hash_run_a':full_hash_a,
    'full_detail_hash_run_b':full_hash_b,
    'full_detail_hash_equal':True
}

print('PASS: deterministic rerun reproduced identical compact and full-detail hashes.')


PASS: deterministic rerun reproduced identical compact and full-detail hashes.


In [10]:
# CODE CELL 10/11 — Transparent protocol amendment and manuscript-safe summary

protocol_amendment={
    'amendment_kind':'POST_FEASIBILITY_VALIDATION_PATH_AMENDMENT',
    'reason':'External blinded adjudication package was prepared in Phase55, but external review is not feasible for the current project.',
    'external_adjudication_executed':False,
    'independently_blinded_external_validation':False,
    'replacement_analysis':'Frozen-method multi-seed synthetic robustness, negative controls, severity characterization, cohort-consequence stability, seed-cluster bootstrap, and deterministic rerun.',
    'method_retuned':False,
    'method_config_sha256':EXPECTED_P42_METHOD_CONFIG,
    'robustness_spec_sha256':ROBUSTNESS_SPEC_SHA,
    'claim_boundary':'Computational robustness only; not independent external validation, official OMOP/MI-CDM execution, or clinical safety validation.'
}
protocol_amendment['amendment_sha256']=canonical_hash(protocol_amendment)

total_cases=sum(r['cases'] for r in seed_rows)
total_truth=sum(r['truth_events'] for r in seed_rows)
total_matched=sum(r['matched_truth_events'] for r in seed_rows)
total_extra=sum(r['unexpected_alerts'] for r in seed_rows)
total_exact=sum(r['exact_case_matches'] for r in seed_rows)
total_norm_expected=sum(r['expected_normalizations'] for r in seed_rows)
total_norm_match=sum(r['matched_normalizations'] for r in seed_rows)

manuscript_text=(
    f'Using the frozen Phase 42 rule set without retuning, the robustness extension evaluated '
    f'{total_cases} newly generated synthetic cases across {len(ROBUSTNESS_SEEDS)} predeclared random seeds, '
    f'containing {total_truth} authored fault events. The frozen detector matched {total_matched} authored events '
    f'and produced {total_extra} unexpected alerts. Exact case-level agreement was '
    f'{total_exact}/{total_cases}. Valid normalization checks matched {total_norm_match}/{total_norm_expected}. '
    f'All predeclared negative controls passed. A fixed severity sweep characterized the existing '
    f'{ABS_TOLERANCE_MM3:g} mm3 absolute numeric tolerance without changing it. Downstream cohort membership '
    f'was re-evaluated for every case and summarized by fault family. Uncertainty was estimated by '
    f'{BOOTSTRAP_REPS} bootstrap resamples clustered by robustness seed. One complete seed was rerun and '
    f'reproduced identical compact and full-detail canonical hashes. These analyses establish repeated '
    f'synthetic robustness and reproducibility only; independent blinded external validation, official '
    f'OMOP/MI-CDM execution, and clinical safety validation were not performed.'
)

protocol_path=STAGE/'ICHI2027_Phase56_PROTOCOL_AMENDMENT.json'
protocol_path.write_text(json.dumps(protocol_amendment,sort_keys=True,indent=2)+'\n',encoding='utf-8')

manuscript_path=STAGE/'ICHI2027_Phase56_MANUSCRIPT_SAFE_RESULTS.txt'
manuscript_path.write_text(manuscript_text+'\n',encoding='utf-8')

print(manuscript_text)


Using the frozen Phase 42 rule set without retuning, the robustness extension evaluated 2748 newly generated synthetic cases across 12 predeclared random seeds, containing 2880 authored fault events. The frozen detector matched 2880 authored events and produced 0 unexpected alerts. Exact case-level agreement was 2748/2748. Valid normalization checks matched 180/180. All predeclared negative controls passed. A fixed severity sweep characterized the existing 1e-06 mm3 absolute numeric tolerance without changing it. Downstream cohort membership was re-evaluated for every case and summarized by fault family. Uncertainty was estimated by 5000 bootstrap resamples clustered by robustness seed. One complete seed was rerun and reproduced identical compact and full-detail canonical hashes. These analyses establish repeated synthetic robustness and reproducibility only; independent blinded external validation, official OMOP/MI-CDM execution, and clinical safety validation were not performed.


In [11]:
# CODE CELL 11/11 — Public outputs, integrity manifest, and shareable summary

seed_path=STAGE/'ICHI2027_Phase56_SEED_SUMMARY.csv'
family_path=STAGE/'ICHI2027_Phase56_FAMILY_SUMMARY.csv'
negative_path=STAGE/'ICHI2027_Phase56_NEGATIVE_CONTROLS.csv'
severity_path=STAGE/'ICHI2027_Phase56_VALUE_SEVERITY_SWEEP.csv'
cohort_path=STAGE/'ICHI2027_Phase56_COHORT_CONSEQUENCES_BY_FAMILY.csv'
bootstrap_path=STAGE/'ICHI2027_Phase56_BOOTSTRAP_CI.csv'
repro_path=STAGE/'ICHI2027_Phase56_REPRODUCIBILITY.json'

write_csv(seed_path,seed_rows)
write_csv(family_path,family_rows)
write_csv(negative_path,negative_rows)
write_csv(severity_path,severity_rows)
write_csv(cohort_path,cohort_family_rows)
write_csv(bootstrap_path,bootstrap_rows)
repro_path.write_text(json.dumps(repro,sort_keys=True,indent=2)+'\n',encoding='utf-8')

readme=STAGE/'ICHI2027_Phase56_README.txt'
readme.write_text(
    'Phase56 is a post-feasibility computational robustness extension using the frozen Phase42 method without retuning.\n'
    'It executes 12 predeclared synthetic seeds, negative controls, a fixed numeric-severity sweep, cohort-consequence analysis, seed-cluster bootstrap, and deterministic rerun.\n'
    'The original Phase43 sealed holdout is reproduced before any new-seed analysis.\n'
    'This phase does not claim independent external validation, official OMOP/MI-CDM execution, real-data fidelity performance, prevalence estimation, or clinical safety.\n',
    encoding='utf-8'
)

artifacts=[
    spec_path,seed_path,family_path,negative_path,severity_path,cohort_path,
    bootstrap_path,repro_path,protocol_path,manuscript_path,readme
]

summary={
    'project':PROJECT,
    'phase':PHASE,
    'run_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
    'input_phase55_shareable_sha256':sha256(P['p55']),
    'phase42_method_config_sha256':EXPECTED_P42_METHOD_CONFIG,
    'phase43_truth_seal_sha256':EXPECTED_TRUTH_SEAL,
    'phase43_truth_seal_reproduced':True,
    'method_retuned':False,
    'robustness_spec_sha256':ROBUSTNESS_SPEC_SHA,
    'robustness_seed_count':len(ROBUSTNESS_SEEDS),
    'robustness_seeds':ROBUSTNESS_SEEDS,
    'new_synthetic_cases':total_cases,
    'authored_fault_events':total_truth,
    'matched_fault_events':total_matched,
    'unexpected_alerts':total_extra,
    'exact_case_matches':total_exact,
    'exact_case_match_rate':total_exact/total_cases,
    'valid_normalizations_expected':total_norm_expected,
    'valid_normalizations_matched':total_norm_match,
    'negative_control_cases':len(negative_rows),
    'negative_control_failures':sum(not r['passes_negative_control'] for r in negative_rows),
    'severity_sweep_levels':len(severity_rows),
    'frozen_abs_tolerance_mm3':ABS_TOLERANCE_MM3,
    'bootstrap_reps':BOOTSTRAP_REPS,
    'bootstrap_cluster_unit':'robustness_seed',
    'deterministic_reproducibility_passed':bool(repro['compact_hash_equal'] and repro['full_detail_hash_equal']),
    'external_reviewer_response_received':False,
    'independently_blinded_external_validation':False,
    'official_omop_etl_executed':False,
    'official_micdm_load_executed':False,
    'ready_for_official_armA':False,
    'real_authorized_mri_processed_by_proposed_fidelity_method':False,
    'validation_scope':'FROZEN_METHOD_MULTI_SEED_SYNTHETIC_ROBUSTNESS_NOT_EXTERNAL_VALIDATION',
    'next_step':'PHASE57_FINAL_EVIDENCE_MATRIX_FIGURES_TABLES_AND_MANUSCRIPT_RESULTS_FREEZE',
    'artifact_sha256':{p.name:sha256(p) for p in artifacts},
    'warning':'Repeated synthetic robustness does not substitute for independent external validation or official target execution.'
}

summary_path=STAGE/'ICHI2027_Phase56_SHAREABLE.json'
summary_path.write_text(json.dumps(summary,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifacts.append(summary_path)

public_zip=STAGE/'ICHI2027_Phase56_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(public_zip,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in artifacts:
        z.write(p,p.name)

OUT.mkdir(parents=True,exist_ok=False)
for p in artifacts+[public_zip]:
    dest=OUT/p.name
    for attempt in range(3):
        try:
            shutil.copyfile(p,dest)
            assert sha256(dest)==sha256(p)
            break
        except Exception as e:
            if attempt==2:
                raise RuntimeError('Drive save/readback verification failed for '+p.name) from e
            time.sleep(1+attempt)

print('SUCCESS: Phase56 computational robustness extension completed.')
print('Seeds:',len(ROBUSTNESS_SEEDS))
print('New synthetic cases:',total_cases)
print('Authored fault events:',total_truth)
print('Matched fault events:',total_matched)
print('Unexpected alerts:',total_extra)
print('Independent external validation: FALSE')
print('Next step:',summary['next_step'])
print('Shareable:',OUT/summary_path.name)
print('Public ZIP:',OUT/public_zip.name)


SUCCESS: Phase56 computational robustness extension completed.
Seeds: 12
New synthetic cases: 2748
Authored fault events: 2880
Matched fault events: 2880
Unexpected alerts: 0
Independent external validation: FALSE
Next step: PHASE57_FINAL_EVIDENCE_MATRIX_FIGURES_TABLES_AND_MANUSCRIPT_RESULTS_FREEZE
Shareable: /content/drive/MyDrive/ICHI2027/Phase56/run_20261001T162938_319424Z/ICHI2027_Phase56_SHAREABLE.json
Public ZIP: /content/drive/MyDrive/ICHI2027/Phase56/run_20261001T162938_319424Z/ICHI2027_Phase56_PUBLIC_OUTPUTS.zip


## After Phase 56

Upload **`ICHI2027_Phase56_SHAREABLE.json`**.

If the frozen method remains stable across all predeclared seeds and controls, Phase 57 should stop adding new experiments and instead freeze the final evidence matrix, figures, tables, manuscript-safe Results language, limitations, and claim boundaries.

The manuscript should continue to state explicitly that external blinded adjudication and official Arm A execution were not completed.
